# 7.9 — Gradiente

**Prerequisiti:** 7.8 Differenziabilità, 6.2 Geometria vettoriale, 5.9 Teoremi fondamentali della derivazione.


## Perché ci interessa

Nel notebook 7.8 abbiamo introdotto il gradiente $\nabla f$ come "il vettore che rende $f$ differenziabile". Ma il gradiente è molto più di un artificio tecnico: la sua direzione ha un significato geometrico preciso — indica la direzione lungo cui $f$ **cresce più velocemente** — e la sua lunghezza misura quanto rapidamente. Questo lo rende lo strumento centrale per l'ottimizzazione (trovare massimi e minimi, notebook 7.12-7.13) e per capire la geometria delle curve di livello (notebook 7.4).

## Teoria, passo dopo passo

### La derivata direzionale

Finora abbiamo derivato solo lungo gli assi coordinati (derivate parziali, notebook 7.7). Generalizziamo: dato un versore $u$ (vettore con $\|u\|=1$, notebook 6.2), la **derivata direzionale** di $f$ in $x_0$ lungo $u$ è:

$$D_uf(x_0) = \lim_{t\to0}\frac{f(x_0+tu)-f(x_0)}t$$

(la pendenza della sezione del grafico lungo la direzione $u$). Per $u=e_1=(1,0,\dots,0)$, si ritrova $D_{e_1}f=\partial f/\partial x_1$: le derivate parziali sono un caso particolare di derivata direzionale.

### Formula del gradiente per la derivata direzionale

> **Teorema.** Se $f$ è differenziabile in $x_0$ (notebook 7.8), allora per ogni versore $u$:
> $$D_uf(x_0) = \nabla f(x_0)\cdot u$$

**Dimostrazione.** Dalla definizione di differenziabilità con $h=tu$ ($\|h\|=|t|\|u\|=|t|$, essendo $u$ un versore): $f(x_0+tu)=f(x_0)+\nabla f(x_0)\cdot(tu)+o(|t|)=f(x_0)+t\,(\nabla f(x_0)\cdot u)+o(|t|)$ (usando la bilinearità del prodotto scalare, notebook 6.20). Quindi $\dfrac{f(x_0+tu)-f(x_0)}t=\nabla f(x_0)\cdot u+\dfrac{o(|t|)}t\to\nabla f(x_0)\cdot u$ per $t\to0$. $\blacksquare$

Questo teorema mostra che, se $f$ è differenziabile, **tutte** le infinite derivate direzionali si ottengono da un unico vettore $\nabla f(x_0)$: non serve calcolare ogni direzione separatamente.

### La direzione di massima crescita

Usando la formula $D_uf(x_0)=\nabla f(x_0)\cdot u=\|\nabla f(x_0)\|\,\|u\|\cos\theta=\|\nabla f(x_0)\|\cos\theta$ (notebook 6.2, essendo $\|u\|=1$, con $\theta$ l'angolo tra $\nabla f(x_0)$ e $u$): questa quantità è **massima** quando $\cos\theta=1$, cioè $\theta=0$, cioè quando $u$ punta nella **stessa direzione di $\nabla f(x_0)$**. In quel caso, $D_uf(x_0)=\|\nabla f(x_0)\|$ (il valore massimo possibile).

> **Il gradiente $\nabla f(x_0)$ punta nella direzione di massima crescita di $f$ in $x_0$, e la sua norma è il tasso di crescita in quella direzione.**

Analogamente, $-\nabla f(x_0)$ (direzione opposta, $\theta=\pi$) è la direzione di massima **decrescita**.

### Il gradiente è perpendicolare alle curve di livello

> **Teorema.** Se $\gamma(t)$ è una curva contenuta nell'insieme di livello $\{f=c\}$ (notebook 7.4) con $\gamma(0)=x_0$, allora $\nabla f(x_0)$ è ortogonale a $\gamma'(0)$ (il vettore tangente alla curva).

**Intuizione della dimostrazione**: $f(\gamma(t))=c$ per ogni $t$ (la curva resta sull'insieme di livello); derivando entrambi i membri rispetto a $t$ in $t=0$ (con la regola della catena in più variabili, che dimostreremo nel notebook 7.20) si ottiene $\nabla f(x_0)\cdot\gamma'(0)=0$: prodotto scalare nullo, cioè ortogonalità (notebook 6.2). Questo spiega perché, sulle mappe topografiche, la direzione di massima pendenza (il gradiente dell'altitudine) è sempre perpendicolare alle curve di livello disegnate.

In [ ]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt

x, y = sp.symbols('x y', real=True)
f = x**2 + y**2

grad_f = sp.Matrix([sp.diff(f, x), sp.diff(f, y)])
print("Gradiente simbolico:", grad_f.T)

# Valutiamo il gradiente in diversi punti e lo confrontiamo con le curve di livello
f_num = sp.lambdify((x, y), f, 'numpy')
gradx_num = sp.lambdify((x, y), grad_f[0], 'numpy')
grady_num = sp.lambdify((x, y), grad_f[1], 'numpy')

X = np.linspace(-2, 2, 20)
Y = np.linspace(-2, 2, 20)
Xg, Yg = np.meshgrid(X, Y)
Zg = f_num(Xg, Yg)
GX, GY = gradx_num(Xg, Yg), grady_num(Xg, Yg)

fig, ax = plt.subplots(figsize=(6, 6))
cs = ax.contour(Xg, Yg, Zg, levels=[0.5, 1, 2, 3, 4], cmap='viridis')
ax.clabel(cs, inline=True, fontsize=8)
ax.quiver(Xg[::2, ::2], Yg[::2, ::2], GX[::2, ::2], GY[::2, ::2], color='red', alpha=0.6)
ax.set_aspect('equal')
ax.set_title("Curve di livello di f=x²+y² e il campo dei gradienti\n(perpendicolari alle curve, verso l'esterno)")
plt.show()

# Verifica: derivata direzionale = grad . u, per un versore u a 45 gradi
punto = (1, 1)
u = sp.Matrix([sp.cos(sp.pi/4), sp.sin(sp.pi/4)])
D_u = (grad_f.subs({x: punto[0], y: punto[1]}).T * u)[0]
print("\nDerivata direzionale a 45° in (1,1):", sp.simplify(D_u))
print("Norma del gradiente in (1,1):", sp.sqrt((grad_f.subs({x:1,y:1}).T * grad_f.subs({x:1,y:1}))[0]))

## Esercizio svolto e commentato

**Problema:** per $f(x,y)=x^2y$, trova la direzione di massima crescita in $(2,1)$ e la derivata direzionale in quella direzione; poi calcola la derivata direzionale nella direzione $u=\left(\frac1{\sqrt2},\frac1{\sqrt2}\right)$.

**Ragionamento:**

**Passo 1 — calcoliamo il gradiente**: $\partial f/\partial x=2xy$, $\partial f/\partial y=x^2$. In $(2,1)$: $\nabla f(2,1)=(2\times2\times1,\,2^2)=(4,4)$.

**Passo 2 — la direzione di massima crescita è quella del gradiente stesso** (per il teorema visto nella teoria): il versore corrispondente è $\dfrac{\nabla f(2,1)}{\|\nabla f(2,1)\|}=\dfrac{(4,4)}{\sqrt{32}}=\left(\dfrac1{\sqrt2},\dfrac1{\sqrt2}\right)$ (semplificando $\sqrt{32}=4\sqrt2$).

**Passo 3 — la derivata direzionale massima è la norma del gradiente**: $\|\nabla f(2,1)\|=\sqrt{16+16}=\sqrt{32}=4\sqrt2$.

**Passo 4 — notiamo che la direzione richiesta nel secondo quesito, $u=\left(\frac1{\sqrt2},\frac1{\sqrt2}\right)$, è esattamente** la direzione di massima crescita appena trovata! Quindi $D_uf(2,1)=4\sqrt2$ direttamente, senza bisogno di ricalcolare il prodotto scalare.

**Conclusione:** in $(2,1)$, $f$ cresce più rapidamente lungo la direzione $\left(\frac1{\sqrt2},\frac1{\sqrt2}\right)$ (a $45°$), con tasso $4\sqrt2\approx5.66$.

**Perché conviene sempre calcolare il gradiente e la sua norma per **prime**, prima di rispondere a domande su derivate direzionali specifiche:** una volta noto $\nabla f(x_0)$, ogni derivata direzionale $D_uf(x_0)$ si ottiene con un semplice prodotto scalare $\nabla f(x_0)\cdot u$ (teorema di questo notebook) — e il caso di massima crescita, spesso richiesto, è semplicemente $\|\nabla f(x_0)\|$ nella direzione del gradiente normalizzato. Calcolare il gradiente una sola volta permette di rispondere a **qualsiasi** domanda direzionale successiva con un calcolo minimo, invece di ripartire dalla definizione con il limite ogni volta.

In [ ]:
import sympy as sp

x, y = sp.symbols('x y', real=True)
f = x**2 * y

grad_f = sp.Matrix([sp.diff(f, x), sp.diff(f, y)])
grad_in_punto = grad_f.subs({x: 2, y: 1})
print("Gradiente in (2,1):", grad_in_punto.T)

norma_grad = sp.sqrt((grad_in_punto.T * grad_in_punto)[0])
print("Norma del gradiente (crescita massima):", norma_grad)

u = sp.Matrix([1/sp.sqrt(2), 1/sp.sqrt(2)])
D_u = (grad_in_punto.T * u)[0]
print("Derivata direzionale lungo u=(1/sqrt2,1/sqrt2):", sp.simplify(D_u))

## Esercizi da risolvere

1. Calcola $\nabla f$ per $f(x,y)=e^x\cos(y)$, e valutalo in $(0,0)$.
2. Trova la derivata direzionale di $f(x,y)=x^2-y^2$ in $(1,1)$ lungo la direzione $u=(0,1)$ (asse $y$) — confronta con $\partial f/\partial y(1,1)$.
3. Per $f(x,y,z)=x^2+y^2+z^2$, trova la direzione di massima decrescita in $(1,1,1)$.
4. *(più impegnativo)* Dimostra che, per $f(x,y)=x^2+y^2$, il gradiente in ogni punto $(x_0,y_0)\ne(0,0)$ punta esattamente **lontano dall'origine** lungo la direzione radiale (suggerimento: confronta $\nabla f(x_0,y_0)$ con il vettore posizione $(x_0,y_0)$).

<details>
<summary><strong>👉 Soluzioni (clicca per visualizzare)</strong></summary>

**1.** $\partial f/\partial x=e^x\cos(y)$, $\partial f/\partial y=-e^x\sin(y)$. In $(0,0)$: $\nabla f(0,0)=(e^0\cos0,\,-e^0\sin0)=(1,0)$.

**2.** $\nabla f(1,1)=(2x,-2y)|_{(1,1)}=(2,-2)$. $D_uf(1,1)=\nabla f(1,1)\cdot(0,1)=2\times0+(-2)\times1=-2$. Coincide con $\partial f/\partial y(1,1)=-2\times1=-2$: coerente, poiché $u=e_2=(0,1)$ è il versore dell'asse $y$, e la derivata direzionale lungo un asse è, per definizione, la derivata parziale corrispondente.

**3.** $\nabla f(1,1,1)=(2,2,2)$: la direzione di massima decrescita è $-\nabla f=(-2,-2,-2)$, cioè il versore $\left(-\frac1{\sqrt3},-\frac1{\sqrt3},-\frac1{\sqrt3}\right)$.

**4.** $\nabla f(x_0,y_0)=(2x_0,2y_0)=2(x_0,y_0)$: è un **multiplo positivo** (fattore $2>0$) del vettore posizione $(x_0,y_0)$ stesso — quindi punta esattamente nella stessa direzione, cioè radialmente **verso l'esterno** (allontanandosi dall'origine, dove $f$ vale $0$, il minimo assoluto). Questo conferma l'intuizione geometrica: $f$ cresce più rapidamente allontanandosi dal centro del paraboloide.

Verifica computazionale:
```python
import sympy as sp

x, y, z = sp.symbols('x y z', real=True)
f1 = sp.exp(x) * sp.cos(y)
print("Es.1:", sp.diff(f1,x).subs({x:0,y:0}), sp.diff(f1,y).subs({x:0,y:0}))

f3 = x**2 + y**2 + z**2
grad3 = sp.Matrix([sp.diff(f3,x), sp.diff(f3,y), sp.diff(f3,z)]).subs({x:1,y:1,z:1})
print("Es.3:", grad3.T)
```

</details>

## Riepilogo finale

- La **derivata direzionale** $D_uf(x_0)$ generalizza le derivate parziali a una direzione qualsiasi $u$ (versore); per funzioni differenziabili, $D_uf(x_0)=\nabla f(x_0)\cdot u$.
- Il **gradiente** $\nabla f(x_0)$ punta nella direzione di **massima crescita** di $f$, con tasso pari a $\|\nabla f(x_0)\|$; $-\nabla f(x_0)$ è la direzione di massima decrescita.
- Il gradiente è sempre **ortogonale** alle curve (o superfici) di livello di $f$: una proprietà geometrica fondamentale, usata costantemente in ottimizzazione.
- Conoscere $\nabla f(x_0)$ permette di calcolare **ogni** derivata direzionale con un semplice prodotto scalare, senza ripartire dalla definizione.

## Collegamenti

Il prossimo notebook è **7.10 — Derivate seconde**: deriviamo le derivate parziali stesse, ottenendo le derivate parziali seconde, e studiamo un risultato notevole (il teorema di Schwarz) sulla loro simmetria.